# CLIP, one small calculation at a time

This notebook has two small parts: reproduce the lecture’s exact 3D cat/dog/car loss, then run the same training step on three pairs with tiny learnable branches. These are **chosen teaching features**, not pretrained CLIP image or text embeddings. No network download is needed after installing PyTorch.

Install once: `%pip install torch` (tested with torch 2.13.0). Run every cell in order.

[Interactive loss tutorial](https://nipunbatra.github.io/interactives/clip-loss/) · [Browser lab](https://nipunbatra.github.io/clip-lab/learn.html) · [CLIP paper, Figure 3](https://proceedings.mlr.press/v139/radford21a.html)

In [1]:
import torch
import torch.nn.functional as F
torch.manual_seed(7)
torch.set_num_threads(1)
torch.set_printoptions(precision=4, sci_mode=False)

## 1. The same three image and caption vectors
Rows are cat, dog and car, exactly as in the lecture and interactive. Start with the chosen raw outputs, then normalize each row. The row order preserves the pairs.

In [2]:
raw_i = torch.tensor([[2., .5, .2], [.4, 2., .1], [.2, .3, 2.]])
raw_t = torch.tensor([[1.6, .8, .4], [.7, 1.4, .3], [.4, .5, 1.5]])
U = F.normalize(raw_i, dim=-1)
V = F.normalize(raw_t, dim=-1)
C = U @ V.T
S = 2 * C                 # teaching scale alpha = 2
print("Cosines:\n", C)
print("Scaled logits:\n", S)

Cosines:
 tensor([[0.9693, 0.6543, 0.3996],
        [0.6091, 0.9556, 0.3933],
        [0.3651, 0.3582, 0.9745]])
Scaled logits:
 tensor([[1.9386, 1.3087, 0.7993],
        [1.2182, 1.9112, 0.7867],
        [0.7302, 0.7163, 1.9490]])


## 2. Each direction has its own denominator
`dim=1` chooses a caption for each image. `dim=0` chooses an image for each caption. Both displays keep images as rows.

In [3]:
image_to_text = S.softmax(dim=1)
text_to_image = S.softmax(dim=0)
print("Image → text shares:\n", image_to_text)
print("Text → image shares:\n", text_to_image)
assert torch.allclose(image_to_text.sum(1), torch.ones(3))
assert torch.allclose(text_to_image.sum(0), torch.ones(3))

Image → text shares:
 tensor([[0.5398, 0.2875, 0.1727],
        [0.2740, 0.5480, 0.1780],
        [0.1862, 0.1837, 0.6301]])
Text → image shares:
 tensor([[0.5601, 0.2959, 0.1944],
        [0.2726, 0.5405, 0.1919],
        [0.1673, 0.1636, 0.6137]])


## 3. Penalize the observed partner, then average
PyTorch cross-entropy accepts **logits**. Do not apply softmax before passing them to `cross_entropy`.

In [4]:
y = torch.arange(3)        # observed partners: (0,0), (1,1), (2,2)
L_image = -image_to_text.diag().log().mean()
L_text = -text_to_image.diag().log().mean()
loss = (L_image + L_text) / 2
compact = (F.cross_entropy(S, y) + F.cross_entropy(S.T, y)) / 2
print("Directional losses:", L_image.item(), L_text.item())
print("Symmetric loss:", loss.item())
assert torch.allclose(loss, compact)
assert abs(loss.item() - 0.5605264020425855) < 1e-6

Directional losses: 0.5600195527076721 0.561033308506012
Symmetric loss: 0.560526430606842


## 4. Give both branches something to learn
For this tiny demonstration, each branch reads a three-number one-hot record ID. A linear layer takes the place of each large encoder. This demonstrates the objective and optimizer; it is **not** a test of visual generalization or a trained CLIP checkpoint.

In [5]:
images = torch.eye(3)     # tiny teaching inputs, not pixels
tokens = torch.eye(3)     # tiny teaching inputs, not language tokens
image_encoder = torch.nn.Linear(3, 2, bias=False)
text_encoder = torch.nn.Linear(3, 2, bias=False)
W_I = torch.nn.Parameter(torch.eye(2))
W_T = torch.nn.Parameter(torch.eye(2))
log_scale = torch.nn.Parameter(torch.tensor(2.).log())
parameters = list(image_encoder.parameters()) + list(text_encoder.parameters())
parameters += [W_I, W_T, log_scale]
optimizer = torch.optim.Adam(parameters, lr=0.03)

def evaluate():
    with torch.no_grad():
        u = F.normalize(image_encoder(images) @ W_I, dim=-1)
        v = F.normalize(text_encoder(tokens) @ W_T, dim=-1)
        s = log_scale.exp() * (u @ v.T)
        y = torch.arange(3)
        return ((F.cross_entropy(s, y) + F.cross_entropy(s.T, y))/2).item()
before = evaluate()
print("Initial loss:", before)

Initial loss: 2.328686237335205


## 5. The training step from the slide
The optimizer sees both encoders, both projections and the log scale. Here is the exact code shown on the lecture slide.

In [6]:
optimizer.zero_grad()
h_I = image_encoder(images)
h_T = text_encoder(tokens)
U = F.normalize(h_I @ W_I, dim=-1)
V = F.normalize(h_T @ W_T, dim=-1)
S = log_scale.exp() * (U @ V.T)
y = torch.arange(len(images), device=S.device)
loss = (F.cross_entropy(S, y)
      + F.cross_entropy(S.T, y)) / 2
loss.backward()
optimizer.step()

## 6. Repeat the same step
All examples here are training pairs. A falling loss on these three IDs shows the mechanics, not transfer to new images.

In [7]:
for _ in range(149):
    optimizer.zero_grad()
    h_I = image_encoder(images)
    h_T = text_encoder(tokens)
    U = F.normalize(h_I @ W_I, dim=-1)
    V = F.normalize(h_T @ W_T, dim=-1)
    S = log_scale.exp() * (U @ V.T)
    y = torch.arange(len(images), device=S.device)
    loss = (F.cross_entropy(S, y)
          + F.cross_entropy(S.T, y)) / 2
    loss.backward()
    optimizer.step()
after = evaluate()
print("Before:", before)
print("After: ", after)
assert after < before
assert after < 0.05

Before: 2.328686237335205
After:  0.0038727668579667807


## Try one change
- Swap two caption records. Keep or change the target indices and explain the difference.
- Change the score scale in the worksheet. Which values stay fixed?
- Remove one inference candidate in the browser lab. Why does the remaining pair’s cosine stay fixed?

For real images, the browser lab loads the pretrained CLIP image and text encoders. The longer [CLIP Lab notebook](https://github.com/nipunbatra/clip-lab/blob/main/notebooks/clip-lab.ipynb) includes real-model inference and the separate color/shape training experiment.

Sources: [Radford et al., 2021, Figure 3](https://proceedings.mlr.press/v139/radford21a.html), [OpenAI CLIP](https://github.com/openai/CLIP). Teaching code and chosen inputs authored for this course.

## One more experiment: watch the vectors move
The [interactive loss tutorial](https://nipunbatra.github.io/interactives/clip-loss/) uses cat, dog and car, with the chosen raw vectors below. It holds τ=0.5 fixed and updates the **vectors themselves**, not encoder parameters. These are the same starting vectors used above. We reproduce the lecture’s first gradient update and 100-update result here.

The tutorial names its cosine matrix `S` and its logits `L`. We keep the lecture's `C` / `S` naming; the calculation is the same.

In [8]:
raw_i = torch.tensor([[2, 0.5, 0.2], [0.4, 2, 0.1], [0.2, 0.3, 2]], dtype=torch.float64, requires_grad=True)
raw_t = torch.tensor([[1.6, 0.8, 0.4], [0.7, 1.4, 0.3], [0.4, 0.5, 1.5]], dtype=torch.float64, requires_grad=True)

def tutorial_loss():
    U = F.normalize(raw_i, dim=-1)
    V = F.normalize(raw_t, dim=-1)
    S = (U @ V.T) / 0.5
    y = torch.arange(3)
    return (F.cross_entropy(S, y) + F.cross_entropy(S.T, y)) / 2

start = tutorial_loss().item()
for step in range(100):
    grad_i, grad_t = torch.autograd.grad(tutorial_loss(), (raw_i, raw_t))
    with torch.no_grad():
        raw_i.add_(grad_i, alpha=-0.25)
        raw_t.add_(grad_t, alpha=-0.25)
    if step == 0:
        first = tutorial_loss().item()
        print("After one update:", first)
        assert abs(first - 0.5397426885763106) < 1e-10
finish = tutorial_loss().item()
print("3D tutorial loss:", start, "→", finish)
assert abs(start - 0.5605264020425855) < 1e-10
assert abs(finish - 0.12493712997191139) < 1e-10


After one update: 0.5397426885763104
3D tutorial loss: 0.5605264020425855 → 0.12493712997191136
